## Setup


In [ ]:
# Install packages
# %pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math
import statistics as stats

In [ ]:
def code_break():
    print("-----------------------------------------")

def get_ssr(matrix: np.ndarray, estimators: np.array):
    ""

In [ ]:
apple = pd.read_csv("APPLE.csv")
apple.head()

## Question 3 ? Manual coefficient estimates


## Estimators

In [ ]:
# 3. 

X = apple[['regprc', 'ecoprc', 'inseason']]
X = sm.add_constant(X)
y = apple['ecolbs']

# ----- Manual Calc ------
X = np.array(X)
XtX = X.T @ X

print(f"XtX: {XtX}")

det_XtX = np.linalg.det(XtX)
# display(f"Determinant XtX: {det_XtX}")
inv_XtX = np.linalg.inv(XtX)

print(f"inv_XtX: {inv_XtX}")

# code_break()

Xty = X.T @ y
# print(Xty)

# code_break()

estimators = inv_XtX @ Xty
# print(estimators)

# estimators_results = zip(['const', 'x1', 'x2', 'x3'], estimators)
estimators_results = pd.DataFrame(estimators, index = ['const', 'x1', 'x2', 'x3'])
estimators_results.rename(mapper={0:'manual'}, axis=1, inplace=True)
estimators_results

## Question 4 ? Homoskedastic covariance and standard errors


## Standard Errors

In [ ]:
# Manual Calculation
beta = np.asarray(estimators).reshape(-1)
e = np.asarray(y).reshape(-1) - X @ beta
e_2 = e**2
ssr = e_2.sum()

s_2 = ssr / (X.shape[0] - X.shape[1])

cov_matrix = s_2*(inv_XtX)
print(cov_matrix)

In [ ]:
manual_s2 = cov_matrix.diagonal()


manual_s2_table = pd.DataFrame(
    manual_s2, 
    index=['const', 'regprc', 'ecoprc', 'inseason'],
    columns=['manual']
)
manual_s2_table

## Question 5 ? Compare with regression output


In [ ]:
model1 = sm.OLS(exog=X, endog=y)
results = model1.fit()

results.summary()

In [ ]:
stats_package_est = pd.DataFrame(results.params)
# comparison_table.reset_index(inplace=True)
stats_package_est.rename(mapper={0:'stats_package'}, inplace=True, axis=1)
stats_package_est

comparison_table = pd.merge(stats_package_est, estimators_results, left_index=True, right_index=True)

display(comparison_table)

In [ ]:
# Compare s2
stats_package_s2 = np.asarray(results.cov_params()).diagonal()
stats_package_s2_table = pd.DataFrame(stats_package_s2, index=['const', 'regprc', 'ecoprc', 'inseason'], columns=['stats_package'])

pd.merge(manual_s2_table, stats_package_s2_table, left_index=True, right_index=True)

## Question 6 ? FWL theorem


## Question 7 ? Model (2) and coefficient interpretation


In [51]:
X = apple[['ecoprc', 'regprc', 'reglbs', 'age', 'faminc', 'hhsize', 'numlt5']]
X = sm.add_constant(X)
y = apple['ecolbs']

In [52]:
model2 = sm.OLS(exog=X, endog=y)
results2 = model2.fit()

results2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 ecolbs   R-squared:                       0.066
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     6.583
Date:                Thu, 19 Nov 2026   Prob (F-statistic):           1.57e-07
Time:                        16:10:39   Log-Likelihood:                -1525.0
No. Observations:                 660   AIC:                             3066.
Df Residuals:                     652   BIC:                             3102.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.7147      0.561      3.056      0.002       0.613       2.817
ecoprc        -2.7526      0.583     -4.722      0.000      -3.897      -1.608
regprc         2.7797      0.705      3.942      0.000       1.395       4.164
reglbs        -0.1386      0.033     -4.192      0.000      -0.203      -0.074
age            0.0005      0.007      0.075      0.940      -0.013       0.014
faminc         0.0031      0.003      1.156      0.248      -0.002       0.008
hhsize         0.1119      0.074      1.509      0.132      -0.034       0.257
numlt5        -0.2027      0.176     -1.155      0.249      -0.547       0.142
==============================================================================
Omnibus:                     1009.586   Durbin-Watson:                   2.018
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           391896.643
Skew:                           8.594   Prob(JB):                         0.00
Kurtosis:                     121.133   Cond. No.                         694.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Question 8 ? Own-price hypothesis test and confidence interval


In [53]:
coefficient = results2.params.loc['ecoprc']
standard_error = results2.cov_params().loc['ecoprc', 'ecoprc']

# Ho: B >= 0, Ha: B < 0; a = 0.05
a = 0.05
test = coefficient / standard_error
critical_value = stats.NormalDist().inv_cdf(a)
if critical_value > test: 
    print(f"Reject the null since {round(test, 5)} < {round(critical_value, 5)}")
else: 
    print("fail to reject")

# Confidence Interval
code_break()
critical_value = -1*stats.NormalDist().inv_cdf(a/2)
# print(critical_value)

lower_bound = coefficient - (critical_value*standard_error)
upper_bound = coefficient + (critical_value*standard_error)

print("CI: ", lower_bound, ",", upper_bound)

Reject the null since -8.09934 < -1.64485
-----------------------------------------
CI:  -3.4186857601763467 , -2.086485425549185


## Question 9 ? HC2 covariance and revised inference


## HC2 Errors

In [54]:
## Diagonal
X_np = np.asarray(X)
inv_XtX = np.linalg.inv(X_np.T @ X_np)
projection_matrix = X_np @ inv_XtX @ X_np.T
# print(projection_matrix)
# code_break()
# print(projection_matrix.shape)
# code_break()
px_diag = projection_matrix.diagonal()

## residuals
u2 = np.asarray((results2.resid)**2)

print(type(px_diag),type(u2))
print(px_diag.shape, u2.shape)

## Omega
omega_weights = np.asarray(results2.resid)**2 / (1 - px_diag)

## HC2 Errors
XtX = X.T @ X
inv_XtX = np.linalg.inv(XtX)

hc2_cov = inv_XtX @ (X_np.T * omega_weights) @ X_np @ inv_XtX
code_break()
pd.DataFrame(hc2_cov, columns=model2.exog_names, index=model2.exog_names)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(660,) (660,)
-----------------------------------------


,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


In [55]:
pd.DataFrame(results2.cov_HC2, columns=model2.exog_names, index=model2.exog_names)

,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


## Question 10 ? Joint F test for the D variables


In [ ]:
X_res_cols = X.columns.to_list()[-5:-1]
X_RES = apple[X_res_cols]
X_RES = sm.add_constant(X_RES)

model_res = sm.OLS(exog=X_RES, endog=y)
results_res = model_res.fit()

print(results_res.summary())
code_break()
print(results2.summary())

# Get SSE for Restricted and Unrestricted
sse_res = sum((y - results_res.predict(X_RES))**2)
sse_unres = sum((y - results2.predict(X))**2)

# F-stat
f_stat = ((sse_res - sse_unres) / len(X_res_cols)) / (sse_unres / (X_RES.shape[0] - len(results2.params) - 1))
code_break()
print(f_stat)

                            OLS Regression Results                            
Dep. Variable:                 ecolbs   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                  0.025
Method:                 Least Squares   F-statistic:                     5.296
Date:                Thu, 19 Nov 2026   Prob (F-statistic):           0.000334
Time:                        16:16:57   Log-Likelihood:                -1537.0
No. Observations:                 660   AIC:                             3084.
Df Residuals:                     655   BIC:                             3106.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.1963      0.444      2.693      0.0